# Pre-processing a .CHA file for use in CE analysis

In [1]:
import os
import sys
import pandas as pd
from tqdm import tqdm

In [2]:
data_path = '../data'
chas_path = os.path.join(data_path, 'chas')
outputs_path = os.path.join(data_path, 'server_ready', 'd_cha_corpus-croatian.tsv')

In [3]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

# Processing all CHA files

Note: the package used here was developed by Prof. Garber. Cite via:

Garber, L. (2019). CHA file python parser. Zenodo. https://doi.org/10.5281/zenodo.3364020

In [4]:
from shared.CHAFile import *

In [5]:
all_files = grab_all_files(chas_path)
len(all_files), all_files[0]

(164, '../data/chas/croation/cro/2014_22_2.cha')

In [6]:
data = []
for f in all_files:
    
    chacha = ChaFile(f)
    meta_data_pieces = f.replace('.cha', '').split('/')[3:]
    
    for line in chacha.getLines():
        line['overlapping_text'] = bool(re.findall(r"(⌋|⌊|⌉|⌈)", line['text']))

        line['corpus'] = '-'.join(meta_data_pieces[:-1])
        
        line['conversation_id'] = '-'.join(meta_data_pieces)
        
        data += [line.copy()]

data = pd.DataFrame(data)

In [7]:
data.shape

(60463, 15)

In [8]:
data.head()

,document_line_no,utterance_no,speaker,text,bullet,recipient,overlapping_text,corpus,conversation_id,com,err,sit,add,act,wor
0,18,1,AEJ,zažella [: zaželjela] van [: vam] je sve najbo...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN
1,20,2,INV,jeste vidili [: vidjeli] one u kninu ?,"[9448, 12046]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN
2,21,3,AEJ,"ajme [/] ajme meni, nemoj mi reći .","[12046, 14676]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN
3,22,4,AEJ,ja sam +...,"[14676, 15737]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN
4,23,5,AEG,"njima su rekli +""/.","[15737, 17215]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
data['conversation_id'].nunique()

164

### Correcting utterances/removing CLAN specific formatting.

In [10]:
def corrected_text(text, contraction_replacement_nonce='CCOONNTTRRAACCTTIIOONN'):
    res = re.sub(r'\(\(.*?\)\)', ' ', str(text))
    # res = re.sub(r'\[.*?\]', ' ', res)
    
    # find contractions and preserve them . . .
    contractions = list(re.findall(r"\w+'\w+", res))
    for contraction in set(contractions):
        replacement = contraction.replace("'", contraction_replacement_nonce)
        res = res.replace(contraction, replacement)
    res = re.sub(r"(⌋|⌊|⌉|⌈)", '', res)
    
    res = res.replace(':', '')
    res = res.replace('\t', ' ')
    
    # remove numbers in parentheses (times???)
    res = re.sub(r'\(\d\.\d\)', ' ', res)
    
    # remove all other special characters.
    res = re.sub(r'[^\w\s]', ' ', res)
    
    res = re.sub(r'\s+', ' ', res).replace('[', ' ').replace(']', ' ').replace(contraction_replacement_nonce, "'")
    
    return res.strip()

In [11]:
data['raw_text'] = data['text'].values
data['text'] = [corrected_text(text) for text in tqdm(data['raw_text'].values)]

100%|██████████| 60463/60463 [00:00<00:00, 161456.91it/s]


In [12]:
data[['corpus', 'raw_text', 'text']].head(n=6)

,corpus,raw_text,text
0,croation-cro,zažella [: zaželjela] van [: vam] je sve najbo...,zažella zaželjela van vam je sve najbolje u no...
1,croation-cro,jeste vidili [: vidjeli] one u kninu ?,jeste vidili vidjeli one u kninu
2,croation-cro,"ajme [/] ajme meni, nemoj mi reći .",ajme ajme meni nemoj mi reći
3,croation-cro,ja sam +...,ja sam
4,croation-cro,"njima su rekli +""/.",njima su rekli
5,croation-cro,"+"" svaka srića [: sreća] !",svaka srića sreća


## Create juxtaposed corpus: (x,y) pairs

In [13]:
max_turns_apart = 30

In [14]:
import warnings; warnings.filterwarnings("ignore")

corpus = []
for cid in tqdm(data['conversation_id'].unique()):
    sub = data.loc[data['conversation_id'].isin([cid])]
    sub_index = sub.index.values
    
    for i in sub_index:
        if i != sub_index[-1]:
            
            # speaker vs. other
            next_line_no = ( (sub_index > i) & (~sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][:(max_turns_apart+1)]
            
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]
            
            # speaker vs. self 
            next_line_no = ( (sub_index > i) & (sub['speaker'].isin([sub['speaker'].loc[i]])) ).values.nonzero()[0]
            next_line_no = sub_index[next_line_no][:(max_turns_apart+1)]
            # next_line_no = next_line_no[next_line_no <= (i + max_turns_apart)]
            for j,li in enumerate(next_line_no):
                d = data.loc[i].to_dict()
                
                d['next_speaker'] = data['speaker'].loc[li]
                d['next_text'] = data['text'].loc[li]
                d['next_utterance_no'] = data['utterance_no'].loc[li]
                d['next_utterance_delta_no'] = j
                
                corpus += [d]

100%|██████████| 164/164 [03:06<00:00,  1.14s/it]


In [15]:
data = pd.DataFrame(corpus)
print(data.shape)
data.head()

(3315889, 20)


,document_line_no,utterance_no,speaker,text,bullet,recipient,overlapping_text,corpus,conversation_id,com,err,sit,add,act,wor,raw_text,next_speaker,next_text,next_utterance_no,next_utterance_delta_no
0,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,zažella [: zaželjela] van [: vam] je sve najbo...,INV,jeste vidili vidjeli one u kninu,2,0
1,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,zažella [: zaželjela] van [: vam] je sve najbo...,AEG,njima su rekli,5,1
2,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,zažella [: zaželjela] van [: vam] je sve najbo...,AEG,svaka srića sreća,6,2
3,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,zažella [: zaželjela] van [: vam] je sve najbo...,AEI,baba ima šezdeset,7,3
4,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,zažella [: zaželjela] van [: vam] je sve najbo...,AEI,ona je još mlada,8,4


In [16]:
# del data['raw_text']

In [17]:
rename_columns = dict()
for col in data.columns:
    if col == 'text':
        rename_columns[col] = 'x_utterance'
    elif col == 'next_text':
        rename_columns[col] = 'y_utterance'
    elif 'next_' in col:
        rename_columns[col]  = col.replace('next', 'y')
    else:
        rename_columns[col] = col

In [18]:
data = data.rename(columns=rename_columns)
data.head()

,document_line_no,utterance_no,speaker,x_utterance,bullet,recipient,overlapping_text,corpus,conversation_id,com,err,sit,add,act,wor,y_speaker,y_utterance,y_utterance_no,y_utterance_delta_no
0,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,INV,jeste vidili vidjeli one u kninu,2,0
1,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,AEG,njima su rekli,5,1
2,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,AEG,svaka srića sreća,6,2
3,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,AEI,baba ima šezdeset,7,3
4,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,AEI,ona je još mlada,8,4


In [19]:
data = data.rename(columns={'speaker':'x_speaker','utterance_no': 'x_turn_id', 'y_utterance_no': 'y_turn_id'})
data.head()

,document_line_no,x_turn_id,x_speaker,x_utterance,bullet,recipient,overlapping_text,corpus,conversation_id,com,err,sit,add,act,wor,y_speaker,y_utterance,y_turn_id,y_utterance_delta_no
0,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,INV,jeste vidili vidjeli one u kninu,2,0
1,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,AEG,njima su rekli,5,1
2,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,AEG,svaka srića sreća,6,2
3,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,AEI,baba ima šezdeset,7,3
4,18,1,AEJ,zažella zaželjela van vam je sve najbolje u no...,"[0, 9448]",ADULT,False,croation-cro,croation-cro-2014_22_2,NaN,NaN,NaN,NaN,NaN,NaN,AEI,ona je još mlada,8,4


In [20]:
data['x_speaker'] = data['corpus'].astype(str) + '-' + data['conversation_id'].astype(str) + '-' + data['x_speaker'].astype(str)

data['y_speaker'] = data['corpus'].astype(str) + '-' + data['conversation_id'].astype(str) + '-' + data['y_speaker'].astype(str)

In [21]:
data['self'] = data['x_speaker'] == data['y_speaker']
data = data.sort_values(by=['corpus', 'conversation_id', 'x_turn_id', 'self', 'y_turn_id'])
data.index = range(len(data))

In [22]:
data[['corpus', 'x_utterance', 'y_utterance']].isna().mean()

corpus         0.0
x_utterance    0.0
y_utterance    0.0
dtype: float64

In [23]:
# # corpus sanity check . . . make sure that conversation_ids are all unique 
# k = data[['conversation_id', 'corpus']].drop_duplicates()
# k['conversation_id'].nunique(), k['conversation_id'].nunique()/len(k)

In [24]:
data[['corpus', 'x_utterance', 'x_speaker', 'y_speaker', 'y_utterance', 'x_turn_id', 'y_turn_id']].head()

,corpus,x_utterance,x_speaker,y_speaker,y_utterance,x_turn_id,y_turn_id
0,croation-cro,e ne govorin govorim kaj d al ne govorin govor...,croation-cro-croation-cro-2011_56-AUN,croation-cro-croation-cro-2011_56-AUM,0 smijeh,1,3
1,croation-cro,e ne govorin govorim kaj d al ne govorin govor...,croation-cro-croation-cro-2011_56-AUN,croation-cro-croation-cro-2011_56-AUM,što to kad si ovu dovodio tamo u ponedjeljak,1,5
2,croation-cro,e ne govorin govorim kaj d al ne govorin govor...,croation-cro-croation-cro-2011_56-AUN,croation-cro-croation-cro-2011_56-AUM,martina martina marina,1,8
3,croation-cro,e ne govorin govorim kaj d al ne govorin govor...,croation-cro-croation-cro-2011_56-AUN,croation-cro-croation-cro-2011_56-AUM,marina,1,10
4,croation-cro,e ne govorin govorim kaj d al ne govorin govor...,croation-cro-croation-cro-2011_56-AUN,croation-cro-croation-cro-2011_56-AUM,još joj ne zna ime ono,1,12


In [25]:
data.shape

(3315889, 20)

## Save outputs for server operations.

In [26]:
data.to_csv(outputs_path, index=False, encoding='utf-8', sep='\t')